# Notebook D v4.6: gold_person_signal_scores + gold_person_scores

**Run order:** After Notebook A, Notebook B, and Notebook 08.

## v4.6 addition: Cell 4 — gold_branch_structural_metrics

Extracts ancestor fill, depth, 3rd cousin, and 4th cousin metrics into a single reusable
gold view. Previously duplicated in two places:
- `_populate_tree_shape_cache()` in `routers/scores.py` (three separate DB queries + Python arithmetic)
- `notebook_scores_snapshot.ipynb` Cell 4 (inline tree_shape, cousin3, cousin4 CTEs)

Both now consume `SELECT * FROM genealogy.gold_branch_structural_metrics`.
Constants baked into the view: FILL_SLOTS=63, DEPTH_CEIL=21, EXPECTED_3RD=22, EXPECTED_4TH=131.

## Architecture

### gold_person_signal_scores
Long-format view — one row per person x signal.
Joins `gold_research_person_signals` (detection booleans + context columns)
against `ref_signal_weights` (applicability flags + base scores).
Evaluates applicability per person per signal.
Exposes: `is_applicable`, `is_fired`, `base_score`, `dimension`.

**Adding a new signal in future requires:**
1. Add boolean column to Notebook B
2. Insert one row into `ref_signal_weights` with applicability flags
3. Add to the UNPIVOT IN (...) list in Cell 1 of this notebook
4. If a new applicability flag type is needed, add a handler to the CASE block

### gold_person_scores
Aggregates signal scores into absolute 0-100 scores per person.
Computes proximity+depth weighted scores for ranking/prioritisation.
story_ready threshold uses absolute scores (quality gate, not ranking).

## Score formula
```
completeness_score = (comp_applicable - comp_fired) / comp_applicable * 100
evidence_score     = (evid_applicable - evid_fired) / evid_applicable * 100
story_score        = PERCENT_RANK() OVER (ORDER BY narr_fired) * 100
overall_score      = completeness * 0.40 + evidence * 0.60
```

## Proximity and depth multiplier values
```
proximity_multiplier (for weighted score display):
  0 -> 1.00, 1 -> 1.25, 2 -> 1.50, 3+ -> 2.00

proximity_priority (for action list ordering — inverted direction):
  0 -> 3.00, 1 -> 2.00, 2 -> 1.25, 3+ -> 1.00

depth_multiplier (for weighted score display):
  <=4 -> 1.00, 5-6 -> 1.10, 7-8 -> 1.20, 9+ -> 1.40

depth_priority (for action list ordering — inverted direction):
  <=4 -> 1.40, 5-6 -> 1.20, 7-8 -> 1.00, 9+ -> 0.80
```


In [0]:
%sql
-- ============================================================
-- CELL 1: gold_person_signal_scores
--
-- One row per person x signal from ref_signal_weights.
-- is_applicable: TRUE if this signal's conditions are met for this person.
-- is_fired:      TRUE if the signal boolean is TRUE in the signals table.
--
-- To add a new signal:
--   1. Add detection boolean to Notebook B
--   2. Add row to ref_signal_weights with applicability flags
--   3. Add one line to the UNPIVOT IN (...) list below
--   4. Done — scoring updates automatically
-- ============================================================

CREATE OR REPLACE VIEW genealogy.gold_person_signal_scores AS

WITH

-- Unpivot wide boolean signals table into long format.
-- INCLUDE NULLS retains FALSE values (default EXCLUDE NULLS would drop them).
-- To add a new signal: add one column name to the IN (...) list.
unpivoted AS (
  SELECT person_gedcom_id, signal_code, is_fired
  FROM genealogy.gold_research_person_signals
  UNPIVOT INCLUDE NULLS (
    is_fired FOR signal_code IN (
      SIGNAL_NO_BIRTH_RECORDED,
      SIGNAL_NO_DEATH_RECORDED,
      SIGNAL_NO_MARRIAGES,
      SIGNAL_NO_CHILDREN,
      SIGNAL_MISSING_PARENT,
      SIGNAL_MISSING_CENSUS_COVERAGE,
      SIGNAL_UNCOVERED_SOURCES,
      SIGNAL_DOCS_NOT_TRANSCRIBED,
      SIGNAL_LATE_LIFE_GAP,
      SIGNAL_EARLY_LIFE_ONLY,
      SIGNAL_CHILD_GAPS,
      SIGNAL_UNCONFIRMED_MILITARY,
      SIGNAL_MISSING_OCCUPATION,
      SIGNAL_MISSING_BURIAL,
      SIGNAL_NO_RESIDENCE,
      SIGNAL_NO_DOCUMENTS_AT_ALL,
      SIGNAL_FACT_CONFLICT,
      SIGNAL_TRANSCRIPT_ONLY_FACTS,
      SIGNAL_VERY_LOW_EVIDENCE_DENSITY,
      SIGNAL_LOW_EVIDENCE_DENSITY,
      SIGNAL_SINGLE_SOURCE_DEPENDENCE,
      SIGNAL_UNSOURCED_FAMILY_EVENTS,
      SIGNAL_INCOMPLETE_NAME,
      SIGNAL_IMPRECISE_DATES,
      SIGNAL_IMPRECISE_PLACES,
      SIGNAL_MIGRANT,
      SIGNAL_CONFIRMED_MILITARY,
      SIGNAL_MULTIPLE_SPOUSES,
      SIGNAL_YOUNG_DEATH,
      SIGNAL_LARGE_FAMILY,
      SIGNAL_NEWSPAPER_MENTION,
      SIGNAL_WILL_OR_PROBATE,
      SIGNAL_STORY_WRITTEN,
      SIGNAL_TRANSCRIPT_RICH,
      SIGNAL_VARIED_OCCUPATIONS,
      SIGNAL_POSSIBLE_RESIDENCE,
      SIGNAL_POSSIBLE_MARRIAGE,
      SIGNAL_POSSIBLE_CHILDREN,
      SIGNAL_DNA_CITATION_MISSING,
      SIGNAL_DNA_PATH_INCOMPLETE,
      SIGNAL_NO_DNA_CORROBORATION,
      SIGNAL_COMMON_ANCESTOR_UNVERIFIED
    )
  )
)

SELECT
  u.person_gedcom_id,
  u.signal_code,
  w.dimension,
  w.category,
  w.intent,
  w.base_score,
  w.reason_label,
  w.outcome_label,
  u.is_fired,

  -- ── Applicability evaluation ──────────────────────────────────────────────
  -- A signal is not applicable if any flag check fails.
  -- Flag checks are evaluated in order; first FALSE wins.
  --
  -- ORDERING NOTE:
  --   requires_blood_relative is checked FIRST, before applies_always.
  --   This ensures signals marked requires_blood_relative=TRUE are suppressed
  --   for non-blood relatives even when applies_always=TRUE is also set.
  --   Signals with applies_always=TRUE and requires_blood_relative=NULL
  --   are unaffected and still apply to everyone.
  CASE

    -- ── Blood relative check ───────────────────────────────
    -- Must come BEFORE applies_always so it can override applies_always=TRUE
    -- for signals that have both flags set.
    -- COALESCE treats NULL is_blood_relative (unbranched people) as non-blood.
    WHEN w.requires_blood_relative = TRUE
      AND COALESCE(s.is_blood_relative, FALSE) = FALSE THEN FALSE

    -- ── applies_always short-circuit ─────────────────────────────────────
    -- If requires_blood_relative didn't exclude this person, applies_always
    -- means all remaining flags are skipped.
    WHEN w.applies_always = TRUE THEN TRUE

    -- ── Birth year range ──────────────────────────────────────────────────
    WHEN w.min_birth_year IS NOT NULL
      AND (s.birth_year IS NULL OR s.birth_year < w.min_birth_year) THEN FALSE
    WHEN w.max_birth_year IS NOT NULL
      AND (s.birth_year IS NULL OR s.birth_year > w.max_birth_year) THEN FALSE

    -- ── Sex restriction ───────────────────────────────────────────────────
    WHEN w.sex_restriction IS NOT NULL
      AND s.sex != w.sex_restriction THEN FALSE

    -- ── Must be expected to have died ─────────────────────────────────────
    -- birth_year <= 1930 AND expected_end_year < current_year
    WHEN w.requires_expected_to_die = TRUE
      AND NOT (
        (s.birth_year IS NULL OR s.birth_year <= 1930)
        AND s.expected_end_year < year(current_date)
      ) THEN FALSE

    -- ── Lifespan threshold ────────────────────────────────────────────────
    -- Used by SIGNAL_NO_RESIDENCE: effective_span_years >= 20
    WHEN w.requires_lifespan_gte IS NOT NULL
      AND COALESCE(s.effective_span_years, 0) < w.requires_lifespan_gte THEN FALSE

    -- ── Survived past 40 ─────────────────────────────────────────────────
    -- Used by SIGNAL_LATE_LIFE_GAP
    WHEN w.requires_survived_past_40 = TRUE
      AND NOT (
        s.birth_year IS NOT NULL
        AND s.expected_end_year >= s.birth_year + 40
      ) THEN FALSE

    -- ── Working age ───────────────────────────────────────────────────────
    -- Used by SIGNAL_MISSING_OCCUPATION: male, end_year >= birth + 18
    WHEN w.requires_working_age = TRUE
      AND NOT (
        s.birth_year IS NOT NULL
        AND s.expected_end_year >= s.birth_year + 18
      ) THEN FALSE

    -- ── At least one expected census year ─────────────────────────────────
    -- Used by SIGNAL_MISSING_CENSUS_COVERAGE
    WHEN w.requires_census_years = TRUE
      AND COALESCE(s.total_expected_censuses, 0) = 0 THEN FALSE

    -- ── Post-1837 birth, death, or marriage ───────────────────────────────
    -- Used by SIGNAL_IMPRECISE_DATES
    WHEN w.requires_post_1837_event = TRUE
      AND NOT (
        COALESCE(s.birth_year, 0) >= 1837
        OR COALESCE(s.death_year, 0) >= 1837
        OR COALESCE(s.earliest_marriage_year, 0) >= 1837
      ) THEN FALSE

    -- ── Has at least one matched document ─────────────────────────────────
    -- Used by SIGNAL_DOCS_NOT_TRANSCRIBED
    WHEN w.requires_has_document = TRUE
      AND s.SIGNAL_NO_DOCUMENTS_AT_ALL = TRUE THEN FALSE

    -- ── Has at least one transcript ───────────────────────────────────────
    -- Used by SIGNAL_FACT_CONFLICT, SIGNAL_TRANSCRIPT_ONLY_FACTS
    WHEN w.requires_transcript = TRUE
      AND s.has_any_transcript = FALSE THEN FALSE

    -- ── Has at least one citation in gold_source_coverage ─────────────────
    -- Used by SIGNAL_UNCOVERED_SOURCES
    WHEN w.requires_citations = TRUE
      AND s.has_citations = FALSE THEN FALSE

    -- ── Has family events (marriages or child births) ──────────────────────
    -- Used by SIGNAL_EARLY_LIFE_ONLY, SIGNAL_POSSIBLE_RESIDENCE
    WHEN w.requires_family_events = TRUE
      AND COALESCE(s.num_marriages, 0) = 0
      AND COALESCE(s.num_child_births, 0) = 0 THEN FALSE

    -- ── Proximity restriction ─────────────────────────────────────────────
    -- Used by SIGNAL_NO_CHILDREN (proximity <= 1),
    -- SIGNAL_NO_DNA_CORROBORATION (proximity <= 0)
    WHEN w.requires_proximity_lte IS NOT NULL
      AND COALESCE(s.proximity, 99) > w.requires_proximity_lte THEN FALSE

    -- ── Death recorded AND post-1837 ──────────────────────────────────────
    -- Used by SIGNAL_MISSING_BURIAL
    WHEN w.requires_death_recorded = TRUE
      AND (s.death_year IS NULL OR s.death_year < 1837) THEN FALSE

    -- ── total_facts threshold ─────────────────────────────────────────────
    -- Used by SIGNAL_SINGLE_SOURCE_DEPENDENCE: total_facts >= 3
    WHEN w.requires_total_facts_gte IS NOT NULL
      AND COALESCE(s.total_facts, 0) < w.requires_total_facts_gte THEN FALSE

    -- ── Suppress for young deaths ─────────────────────────────────────────
    -- Used by SIGNAL_NO_MARRIAGES, SIGNAL_NO_CHILDREN
    WHEN w.not_young_death = TRUE
      AND s.death_year IS NOT NULL
      AND s.effective_span_years BETWEEN 16 AND 40 THEN FALSE

    -- ── DNA applicability flags ──────────────────────────────

    -- requires_dna_match_tag: only applicable for DNA Match tagged people.
    -- Used by SIGNAL_DNA_CITATION_MISSING and SIGNAL_DNA_PATH_INCOMPLETE.
    WHEN w.requires_dna_match_tag = TRUE
      AND COALESCE(s.has_dna_match_tag, FALSE) = FALSE THEN FALSE

    -- requires_dna_ancestor_tag: only applicable for Common DNA Ancestor tagged people.
    -- Used by SIGNAL_COMMON_ANCESTOR_UNVERIFIED.
    WHEN w.requires_dna_ancestor_tag = TRUE
      AND COALESCE(s.has_dna_ancestor_tag, FALSE) = FALSE THEN FALSE

    -- requires_generation_lte: only applicable if generation depth <= N.
    -- Used by SIGNAL_NO_DNA_CORROBORATION (gen <= 6).
    -- NULL generation_depth = not in gold_generation_depth = not applicable.
    WHEN w.requires_generation_lte IS NOT NULL
      AND COALESCE(s.generation_depth, 999) > w.requires_generation_lte THEN FALSE

    ELSE TRUE
  END AS is_applicable

FROM unpivoted u
JOIN genealogy.ref_signal_weights w
  ON w.signal_code = u.signal_code
JOIN genealogy.gold_research_person_signals s
  ON s.person_gedcom_id = u.person_gedcom_id;


In [0]:
%sql
-- ============================================================
-- CELL 2: gold_person_scores
-- ============================================================

CREATE OR REPLACE VIEW genealogy.gold_person_scores AS

WITH scores AS (
  SELECT
    person_gedcom_id,

    -- Completeness: applicable max and fired sum
    SUM(CASE WHEN dimension = 'completeness' AND is_applicable AND base_score > 0
             THEN base_score ELSE 0 END) AS comp_applicable,
    SUM(CASE WHEN dimension = 'completeness' AND is_applicable AND is_fired AND base_score > 0
             THEN base_score ELSE 0 END) AS comp_fired,

    -- Evidence: applicable max and fired sum
    SUM(CASE WHEN dimension = 'evidence' AND is_applicable AND base_score > 0
             THEN base_score ELSE 0 END) AS evid_applicable,
    SUM(CASE WHEN dimension = 'evidence' AND is_applicable AND is_fired AND base_score > 0
             THEN base_score ELSE 0 END) AS evid_fired,

    -- Narrative: exclude SIGNAL_STORY_WRITTEN from denominator entirely
    SUM(CASE WHEN dimension = 'narrative' AND is_applicable AND base_score > 0
               AND signal_code != 'SIGNAL_STORY_WRITTEN'
             THEN base_score ELSE 0 END) AS narr_applicable,
    SUM(CASE WHEN dimension = 'narrative' AND is_applicable AND is_fired AND base_score > 0
               AND signal_code != 'SIGNAL_STORY_WRITTEN'
             THEN base_score ELSE 0 END) AS narr_fired,

    -- Story written flag (sourced from signal firing, used as fallback)
    MAX(CASE WHEN signal_code = 'SIGNAL_STORY_WRITTEN' AND is_fired THEN TRUE ELSE FALSE END)
      AS story_written_flag

  FROM genealogy.gold_person_signal_scores
  GROUP BY person_gedcom_id
),

-- Minimum ancestral_proximity per person across all paths to researcher
prox AS (
  SELECT person_id, MIN(ancestral_proximity) AS proximity
  FROM genealogy.gold_ancestral_proximity
  GROUP BY person_id
),

story AS (
  SELECT person_gedcom_id, story_written, story_title, story_doc_id
  FROM genealogy.silver_person_story_status
),

-- depth from gold_research_person_signals
depth_cte AS (
  SELECT person_gedcom_id, depth
  FROM genealogy.gold_research_person_signals
),

multipliers AS (
  SELECT
    pr.person_id AS person_gedcom_id,

    -- Proximity multiplier for weighted score display
    CASE
      WHEN pr.proximity = 0 THEN 1.00
      WHEN pr.proximity = 1 THEN 1.25
      WHEN pr.proximity = 2 THEN 1.50
      ELSE                       2.00
    END AS proximity_multiplier,

    -- Inverted proximity for action list priority ordering
    CASE
      WHEN pr.proximity = 0 THEN 3.00
      WHEN pr.proximity = 1 THEN 2.00
      WHEN pr.proximity = 2 THEN 1.25
      ELSE                       1.00
    END AS proximity_priority,

    -- Depth multiplier for weighted score display
    CASE
      WHEN d.depth <= 4              THEN 1.00
      WHEN d.depth BETWEEN 5 AND 6   THEN 1.10
      WHEN d.depth BETWEEN 7 AND 8   THEN 1.20
      ELSE                                1.40
    END AS depth_multiplier,

    -- Inverted depth for action list priority ordering
    CASE
      WHEN d.depth <= 4              THEN 1.40
      WHEN d.depth BETWEEN 5 AND 6   THEN 1.20
      WHEN d.depth BETWEEN 7 AND 8   THEN 1.00
      ELSE                                0.80
    END AS depth_priority

  FROM prox pr
  LEFT JOIN depth_cte d ON d.person_gedcom_id = pr.person_id
),

base AS (
  SELECT
    p.person_gedcom_id,
    p.given_name,
    p.first_name,
    p.surname,
    p.display_name,
    p.birth_year,
    p.death_year,
    p.sex,
    sc.comp_applicable,
    sc.comp_fired,
    sc.evid_applicable,
    sc.evid_fired,
    sc.narr_applicable,
    sc.narr_fired,
    sc.story_written_flag,
    COALESCE(st.story_written, FALSE)  AS story_written,
    st.story_title,
    st.story_doc_id,
    pr.proximity                       AS ancestral_proximity,
    b.branch,

    COALESCE(m.proximity_multiplier, 0.80) AS proximity_multiplier,
    COALESCE(m.depth_multiplier,     1.00) AS depth_multiplier,
    COALESCE(m.proximity_priority,   1.00) AS proximity_priority,
    COALESCE(m.depth_priority,       0.80) AS depth_priority,

    -- Absolute scores
    ROUND((sc.comp_applicable - sc.comp_fired) / NULLIF(sc.comp_applicable, 0) * 100, 0)
      AS completeness_score,
    ROUND((sc.evid_applicable - sc.evid_fired) / NULLIF(sc.evid_applicable, 0) * 100, 0)
      AS evidence_score,
    ROUND(
      ROUND((sc.comp_applicable - sc.comp_fired) / NULLIF(sc.comp_applicable, 0) * 100, 0) * 0.40
      + ROUND((sc.evid_applicable - sc.evid_fired) / NULLIF(sc.evid_applicable, 0) * 100, 0) * 0.60
    , 0) AS overall_score,

    -- Weighted scores (for ORDER BY only — not displayed in UI)
    ROUND(
      ROUND((sc.comp_applicable - sc.comp_fired) / NULLIF(sc.comp_applicable, 0) * 100, 0)
      * COALESCE(m.proximity_multiplier, 2.00)
      * COALESCE(m.depth_multiplier,     1.40)
    , 1) AS weighted_completeness_score,
    ROUND(
      ROUND((sc.evid_applicable - sc.evid_fired) / NULLIF(sc.evid_applicable, 0) * 100, 0)
      * COALESCE(m.proximity_multiplier, 2.00)
      * COALESCE(m.depth_multiplier,     1.40)
    , 1) AS weighted_evidence_score,
    ROUND(
      (
        ROUND((sc.comp_applicable - sc.comp_fired) / NULLIF(sc.comp_applicable, 0) * 100, 0) * 0.40
        + ROUND((sc.evid_applicable - sc.evid_fired) / NULLIF(sc.evid_applicable, 0) * 100, 0) * 0.60
      )
      * COALESCE(m.proximity_multiplier, 2.00)
      * COALESCE(m.depth_multiplier,     1.40)
    , 1) AS weighted_overall_score

  FROM genealogy.gold_person_life p
  JOIN scores sc                            ON sc.person_gedcom_id = p.person_gedcom_id
  LEFT JOIN genealogy.gold_person_branch b  ON b.person_gedcom_id  = p.person_gedcom_id
  LEFT JOIN prox pr                         ON pr.person_id         = p.person_gedcom_id
  LEFT JOIN story st                        ON st.person_gedcom_id  = p.person_gedcom_id
  LEFT JOIN multipliers m                   ON m.person_gedcom_id   = p.person_gedcom_id
),

ranked AS (
  SELECT
    person_gedcom_id,
    ROUND(PERCENT_RANK() OVER (ORDER BY narr_fired) * 100, 0) AS story_potential_score
  FROM base
)

SELECT
  b.person_gedcom_id,
  b.given_name,
  b.first_name,
  b.surname,
  b.display_name,
  b.birth_year,
  b.death_year,
  b.branch,
  b.sex,

  b.comp_applicable,
  b.comp_fired,
  b.evid_applicable,
  b.evid_fired,
  b.narr_applicable,
  b.narr_fired,

  b.completeness_score,
  b.evidence_score,
  b.overall_score,

  CASE WHEN b.story_written_flag OR b.story_written
       THEN 0
       ELSE r.story_potential_score
  END AS story_potential_score,

  CASE
    WHEN b.story_written_flag OR b.story_written THEN FALSE
    WHEN r.story_potential_score >= 90 AND b.overall_score >= 70 THEN TRUE
    ELSE FALSE
  END AS story_ready,

  b.weighted_completeness_score,
  b.weighted_evidence_score,
  b.weighted_overall_score,

  CASE WHEN b.story_written_flag OR b.story_written
       THEN 0
       ELSE ROUND(
         r.story_potential_score
         * (2 - b.proximity_multiplier)
         * (2 - b.depth_multiplier)
       , 1)
  END AS weighted_story_potential_score,

  b.proximity_multiplier,
  b.depth_multiplier,
  b.proximity_priority,
  b.depth_priority,
  (b.proximity_priority * b.depth_priority) AS person_priority,

  b.story_written,
  b.story_title,
  b.story_doc_id,

  b.ancestral_proximity,
  CASE
    WHEN b.ancestral_proximity = 0             THEN 'Direct Ancestor'
    WHEN b.ancestral_proximity = 1             THEN 'Close'
    WHEN b.ancestral_proximity BETWEEN 2 AND 3 THEN 'Collateral'
    ELSE                                            'Distant'
  END AS proximity_label

FROM base b
JOIN ranked r ON r.person_gedcom_id = b.person_gedcom_id;


In [0]:
%sql
-- ============================================================
-- CELL 3: gold_branch_scores
-- ============================================================

CREATE OR REPLACE VIEW genealogy.gold_branch_scores AS
SELECT
  branch,
  COUNT(*)                                                        AS total_individuals,
  ROUND(AVG(completeness_score))                                  AS avg_completeness,
  ROUND(AVG(evidence_score))                                      AS avg_evidence,
  ROUND(AVG(story_potential_score))                               AS avg_story_potential,
  ROUND(AVG(overall_score))                                       AS avg_overall,
  SUM(CASE WHEN story_written            THEN 1 ELSE 0 END)       AS stories_written,
  SUM(CASE WHEN story_ready
            AND NOT story_written        THEN 1 ELSE 0 END)       AS stories_ready,
  COUNT(CASE WHEN ancestral_proximity = 0 THEN 1 END)             AS direct_ancestors,
  ROUND(AVG(CASE WHEN ancestral_proximity = 0
                 THEN overall_score END))                         AS ancestor_avg_overall,
  MIN(completeness_score)                                         AS min_completeness,
  MIN(evidence_score)                                             AS min_evidence
FROM genealogy.gold_person_scores
WHERE branch IS NOT NULL
GROUP BY branch
ORDER BY avg_overall DESC;


In [0]:
%sql
-- ============================================================
-- CELL 4: gold_branch_structural_metrics
--
-- One row per branch. Combines ancestor fill/depth metrics and
-- 3rd/4th cousin metrics into a single reusable gold view.
--
-- Replaces duplicated logic previously in:
--   - _populate_tree_shape_cache() in routers/scores.py
--   - notebook_scores_snapshot.ipynb Cell 4
--
-- Constants baked in as SQL expressions:
--   63  = per-branch theoretical max ancestor slots gen 3-8 (1+2+4+8+16+32)
--   21  = deepest plausible genealogical line (depth ceiling)
--   22  = expected 3rd cousins per branch (UK avg ~175 / 8 branches)
--   131 = expected 4th cousins per branch (UK avg ~1050 / 8 branches)
--
-- gold_generation_depth.person_id holds GEDCOM IDs matching
-- gold_person_branch.person_gedcom_id (column rename pending).
-- ============================================================

CREATE OR REPLACE VIEW genealogy.gold_branch_structural_metrics AS

WITH

-- ── Ancestor fill and depth ───────────────────────────────────────────────────
-- Gen 0 = researcher, gen 1-2 = branch-ambiguous, excluded.
-- Gen 3-8 are unambiguously branch-specific ancestors.
ancestor_shape AS (
    SELECT
        gpb.branch,
        COUNT(DISTINCT CASE WHEN ggd.generation_depth BETWEEN 3 AND 8
                            THEN ggd.person_id END) AS ancestors_found,
        MAX(ggd.generation_depth)                   AS max_depth
    FROM genealogy.gold_generation_depth ggd
    JOIN genealogy.gold_person_branch gpb
        ON gpb.person_gedcom_id = ggd.person_id
    WHERE ggd.generation_depth >= 3
    GROUP BY gpb.branch
),

-- ── 3rd cousin metrics ────────────────────────────────────────────────────────
-- Common ancestor: gen-4 (great-great-grandparent).
-- Sibling: gen-3 child of gen-4 ancestor NOT on the direct line.
-- 3rd cousin: grandchild of the sibling (two hops down from sibling).
gen4_ancestors AS (
    SELECT person_id AS gggg_id, path[3] AS direct_gen3_id
    FROM genealogy.gold_generation_depth
    WHERE generation_depth = 4
),
gen4_families AS (
    SELECT g.gggg_id, g.direct_gen3_id, f.family_gedcom_id
    FROM gen4_ancestors g
    JOIN genealogy.silver_family f
        ON f.husband_gedcom_id = g.gggg_id OR f.wife_gedcom_id = g.gggg_id
),
collateral_gen3 AS (
    SELECT DISTINCT fc.child_gedcom_id AS collateral_id, gf.gggg_id
    FROM gen4_families gf
    JOIN genealogy.silver_family_child fc ON fc.family_gedcom_id = gf.family_gedcom_id
    WHERE fc.child_gedcom_id != gf.direct_gen3_id
),
collateral_gen3_children AS (
    SELECT DISTINCT fc.child_gedcom_id AS child_id,
                    c3.collateral_id   AS sibling_id, c3.gggg_id
    FROM collateral_gen3 c3
    JOIN genealogy.silver_family f
        ON f.husband_gedcom_id = c3.collateral_id OR f.wife_gedcom_id = c3.collateral_id
    JOIN genealogy.silver_family_child fc ON fc.family_gedcom_id = f.family_gedcom_id
),
third_cousins AS (
    SELECT DISTINCT fc.child_gedcom_id AS third_cousin_id,
                    cgc.sibling_id, cgc.gggg_id
    FROM collateral_gen3_children cgc
    JOIN genealogy.silver_family f
        ON f.husband_gedcom_id = cgc.child_id OR f.wife_gedcom_id = cgc.child_id
    JOIN genealogy.silver_family_child fc ON fc.family_gedcom_id = f.family_gedcom_id
),
cousin3_raw AS (
    SELECT
        gpb.branch,
        COUNT(DISTINCT c3.collateral_id)                              AS known_siblings_3rd,
        COUNT(DISTINCT CASE WHEN tc.third_cousin_id IS NOT NULL
                            THEN c3.collateral_id END)               AS siblings_with_cousins_3rd,
        COUNT(DISTINCT tc.third_cousin_id)                            AS cousins_found_3rd
    FROM collateral_gen3 c3
    JOIN genealogy.gold_person_branch gpb ON gpb.person_gedcom_id = c3.collateral_id
    LEFT JOIN third_cousins tc ON tc.sibling_id = c3.collateral_id
    GROUP BY gpb.branch
),

-- ── 4th cousin metrics ────────────────────────────────────────────────────────
-- Common ancestor: gen-5 (great-great-great-grandparent).
-- Same fan-out pattern, four hops down from sibling.
gen5_ancestors AS (
    SELECT person_id AS ggggg_id, path[4] AS direct_gen4_id
    FROM genealogy.gold_generation_depth
    WHERE generation_depth = 5
),
gen5_families AS (
    SELECT g.ggggg_id, g.direct_gen4_id, f.family_gedcom_id
    FROM gen5_ancestors g
    JOIN genealogy.silver_family f
        ON f.husband_gedcom_id = g.ggggg_id OR f.wife_gedcom_id = g.ggggg_id
),
collateral_gen4 AS (
    SELECT DISTINCT fc.child_gedcom_id AS collateral_id, gf.ggggg_id
    FROM gen5_families gf
    JOIN genealogy.silver_family_child fc ON fc.family_gedcom_id = gf.family_gedcom_id
    WHERE fc.child_gedcom_id != gf.direct_gen4_id
),
c4g3 AS (
    SELECT DISTINCT fc.child_gedcom_id AS collateral_id,
                    c4.ggggg_id, c4.collateral_id AS sibling_id
    FROM collateral_gen4 c4
    JOIN genealogy.silver_family f
        ON f.husband_gedcom_id = c4.collateral_id OR f.wife_gedcom_id = c4.collateral_id
    JOIN genealogy.silver_family_child fc ON fc.family_gedcom_id = f.family_gedcom_id
),
c4g2 AS (
    SELECT DISTINCT fc.child_gedcom_id AS collateral_id, c.ggggg_id, c.sibling_id
    FROM c4g3 c
    JOIN genealogy.silver_family f
        ON f.husband_gedcom_id = c.collateral_id OR f.wife_gedcom_id = c.collateral_id
    JOIN genealogy.silver_family_child fc ON fc.family_gedcom_id = f.family_gedcom_id
),
c4g1 AS (
    SELECT DISTINCT fc.child_gedcom_id AS collateral_id, c.ggggg_id, c.sibling_id
    FROM c4g2 c
    JOIN genealogy.silver_family f
        ON f.husband_gedcom_id = c.collateral_id OR f.wife_gedcom_id = c.collateral_id
    JOIN genealogy.silver_family_child fc ON fc.family_gedcom_id = f.family_gedcom_id
),
fourth_cousins AS (
    SELECT DISTINCT fc.child_gedcom_id AS fourth_cousin_id, c.sibling_id, c.ggggg_id
    FROM c4g1 c
    JOIN genealogy.silver_family f
        ON f.husband_gedcom_id = c.collateral_id OR f.wife_gedcom_id = c.collateral_id
    JOIN genealogy.silver_family_child fc ON fc.family_gedcom_id = f.family_gedcom_id
),
cousin4_raw AS (
    SELECT
        gpb.branch,
        COUNT(DISTINCT c4.collateral_id)                              AS known_siblings_4th,
        COUNT(DISTINCT CASE WHEN fc4.fourth_cousin_id IS NOT NULL
                            THEN c4.collateral_id END)               AS siblings_with_cousins_4th,
        COUNT(DISTINCT fc4.fourth_cousin_id)                          AS cousins_found_4th
    FROM collateral_gen4 c4
    JOIN genealogy.gold_person_branch gpb ON gpb.person_gedcom_id = c4.collateral_id
    LEFT JOIN fourth_cousins fc4 ON fc4.sibling_id = c4.collateral_id
    GROUP BY gpb.branch
),

-- Use gold_person_branch as spine so all 8 branches are always present
all_branches AS (
    SELECT DISTINCT branch
    FROM genealogy.gold_person_branch
    WHERE branch IS NOT NULL
)

SELECT
    ab.branch,

    -- Ancestor fill metrics
    COALESCE(a.ancestors_found, 0)                           AS ancestors_found,
    ROUND(COALESCE(a.ancestors_found, 0) / 63.0 * 100, 1)   AS ancestor_fill_score,
    COALESCE(a.max_depth, 0)                                 AS max_depth,
    ROUND(COALESCE(a.max_depth, 0) / 21.0 * 100, 1)         AS depth_score,

    -- 3rd cousin metrics
    COALESCE(c3.known_siblings_3rd, 0)                       AS known_siblings_3rd,
    COALESCE(c3.cousins_found_3rd, 0)                        AS cousins_found_3rd,
    COALESCE(c3.siblings_with_cousins_3rd, 0)                AS siblings_with_cousins_3rd,
    ROUND(
        COALESCE(c3.siblings_with_cousins_3rd, 0)
        / NULLIF(c3.known_siblings_3rd, 0) * 100, 1
    )                                                        AS sibling_coverage_3rd,
    ROUND(COALESCE(c3.cousins_found_3rd, 0) / 22.0 * 100, 1) AS fill_rate_3rd,

    -- 4th cousin metrics
    COALESCE(c4.known_siblings_4th, 0)                       AS known_siblings_4th,
    COALESCE(c4.cousins_found_4th, 0)                        AS cousins_found_4th,
    COALESCE(c4.siblings_with_cousins_4th, 0)                AS siblings_with_cousins_4th,
    ROUND(
        COALESCE(c4.siblings_with_cousins_4th, 0)
        / NULLIF(c4.known_siblings_4th, 0) * 100, 1
    )                                                        AS sibling_coverage_4th,
    ROUND(COALESCE(c4.cousins_found_4th, 0) / 131.0 * 100, 1) AS fill_rate_4th

FROM all_branches ab
LEFT JOIN ancestor_shape  a  ON a.branch  = ab.branch
LEFT JOIN cousin3_raw     c3 ON c3.branch = ab.branch
LEFT JOIN cousin4_raw     c4 ON c4.branch = ab.branch
ORDER BY ab.branch;

In [0]:
%sql
-- ── Standard denominator checks (all should return 0) ────────────────────────
SELECT 'comp_fired > comp_applicable'  AS check_name, COUNT(*) AS violations
FROM genealogy.gold_person_scores WHERE comp_fired > comp_applicable
UNION ALL
SELECT 'evid_fired > evid_applicable',  COUNT(*) FROM genealogy.gold_person_scores WHERE evid_fired > evid_applicable
UNION ALL
SELECT 'narr_fired > narr_applicable',  COUNT(*) FROM genealogy.gold_person_scores WHERE narr_fired > narr_applicable
UNION ALL
SELECT 'NULL completeness_score',       COUNT(*) FROM genealogy.gold_person_scores WHERE completeness_score IS NULL
UNION ALL
SELECT 'NULL evidence_score',           COUNT(*) FROM genealogy.gold_person_scores WHERE evidence_score IS NULL
UNION ALL
SELECT 'NULL story_potential_score',    COUNT(*) FROM genealogy.gold_person_scores WHERE story_potential_score IS NULL
UNION ALL
SELECT 'comp_applicable = 0',           COUNT(*) FROM genealogy.gold_person_scores WHERE comp_applicable = 0
UNION ALL
SELECT 'evid_applicable = 0',           COUNT(*) FROM genealogy.gold_person_scores WHERE evid_applicable = 0;

-- ── Blood relative score impact check ──────────────────────────
-- Validate that non-blood relatives have reduced applicable maximums.
--
-- Expected results:
--   Blood relatives: higher avg comp_applicable and evid_applicable
--   Non-blood (spouses/in-laws): lower avg comp_applicable and evid_applicable
--     comp_applicable for non-blood: only NO_BIRTH(30) + NO_DEATH(20) +
--       NO_MARRIAGES(15) + NO_CHILDREN(10 if prox<=1) + MISSING_PARENT(25) = ~70-100
--     evid_applicable for non-blood: only INCOMPLETE_NAME(10) +
--       DNA signals if tagged = ~10-30
--   NULL is_blood_relative (unbranched): treated as non-blood, similar low applicables
SELECT
  CASE
    WHEN s.is_blood_relative = TRUE  THEN 'blood relative'
    WHEN s.is_blood_relative = FALSE THEN 'spouse / in-law'
    ELSE                                  'unbranched (NULL)'
  END AS relationship_type,
  COUNT(*)                          AS people,
  ROUND(AVG(ps.comp_applicable), 0) AS avg_comp_applicable,
  ROUND(AVG(ps.evid_applicable), 0) AS avg_evid_applicable,
  ROUND(AVG(ps.completeness_score), 1) AS avg_completeness,
  ROUND(AVG(ps.evidence_score), 1)     AS avg_evidence,
  ROUND(AVG(ps.overall_score), 1)      AS avg_overall
FROM genealogy.gold_person_scores ps
JOIN genealogy.gold_research_person_signals s ON s.person_gedcom_id = ps.person_gedcom_id
GROUP BY
  CASE
    WHEN s.is_blood_relative = TRUE  THEN 'blood relative'
    WHEN s.is_blood_relative = FALSE THEN 'spouse / in-law'
    ELSE                                  'unbranched (NULL)'
  END
ORDER BY avg_comp_applicable DESC;


In [0]:
%sql
SELECT
  'completeness'    AS dimension,
  ROUND(MIN(completeness_score))                        AS min,
  ROUND(PERCENTILE(completeness_score, 0.25))           AS p25,
  ROUND(PERCENTILE(completeness_score, 0.50))           AS median,
  ROUND(PERCENTILE(completeness_score, 0.75))           AS p75,
  ROUND(MAX(completeness_score))                        AS max,
  ROUND(AVG(completeness_score), 1)                     AS mean,
  SUM(CASE WHEN completeness_score = 100 THEN 1 END)    AS at_100,
  SUM(CASE WHEN completeness_score = 0   THEN 1 END)    AS at_0
FROM genealogy.gold_person_scores
UNION ALL
SELECT 'evidence',
  ROUND(MIN(evidence_score)), ROUND(PERCENTILE(evidence_score, 0.25)),
  ROUND(PERCENTILE(evidence_score, 0.50)), ROUND(PERCENTILE(evidence_score, 0.75)),
  ROUND(MAX(evidence_score)), ROUND(AVG(evidence_score), 1),
  SUM(CASE WHEN evidence_score = 100 THEN 1 END),
  SUM(CASE WHEN evidence_score = 0   THEN 1 END)
FROM genealogy.gold_person_scores
UNION ALL
SELECT 'story_potential',
  ROUND(MIN(story_potential_score)), ROUND(PERCENTILE(story_potential_score, 0.25)),
  ROUND(PERCENTILE(story_potential_score, 0.50)), ROUND(PERCENTILE(story_potential_score, 0.75)),
  ROUND(MAX(story_potential_score)), ROUND(AVG(story_potential_score), 1),
  SUM(CASE WHEN story_potential_score = 100 THEN 1 END),
  SUM(CASE WHEN story_potential_score = 0   THEN 1 END)
FROM genealogy.gold_person_scores
ORDER BY dimension;


In [0]:
%sql
SELECT * FROM genealogy.gold_branch_scores;


In [0]:
%sql
-- Spot-check: confirm direct ancestors rank above distant relatives.
-- ASC = lowest weighted score = most research needed.
SELECT
  given_name,
  surname,
  birth_year,
  branch,
  proximity_label,
  ancestral_proximity,
  proximity_multiplier,
  depth_multiplier,
  overall_score,
  weighted_overall_score,
  weighted_story_potential_score
FROM genealogy.gold_person_scores
ORDER BY weighted_overall_score ASC
LIMIT 20;


In [0]:
%sql
-- Cross-join on thresholds to find the right story_ready cutoff.
-- Aim for ~50 story-ready individuals with good direct-ancestor coverage.
-- To tweak the threshold edit story_ready logic in cell 2 (line 200)
WITH thresholds AS (
  SELECT explode(sequence(60, 100, 1)) AS threshold
),
scores AS (
  SELECT person_gedcom_id, story_potential_score, weighted_story_potential_score,
         story_written, proximity_label, branch
  FROM genealogy.gold_person_scores
  WHERE NOT COALESCE(story_written, FALSE)
)
SELECT
  t.threshold,
  COUNT(*)                                                                 AS total_story_ready,
  SUM(CASE WHEN s.proximity_label = 'Direct Ancestor' THEN 1 ELSE 0 END)  AS direct_ancestors,
  COUNT(DISTINCT s.branch)                                                 AS branches_covered
FROM thresholds t
JOIN scores s ON s.weighted_story_potential_score >= t.threshold
GROUP BY t.threshold
ORDER BY t.threshold;
